In [41]:
import cv2
from pynq.lib.video import *
import numpy as np
import time
from pynq import Overlay, MMIO, allocate

ol = Overlay("./AES_SYS.bit", download = True)
print(ol.ip_dict.keys())

NUM_CHUNKS = 12
PIXELS_PER_CHUNK = 320 * 240


dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_gpio_2', 'axi_gpio_3', 'axi_gpio_4', 'axi_gpio_5', 'axi_gpio_6', 'axi_gpio_7', 'axi_gpio_8', 'axi_gpio_9', 'axi_intc_0', 'axi_vdma_0', 'axi_cdma_0', 'processing_system7_0'])


In [42]:
GPIO0_ADDR = ol.ip_dict['axi_gpio_0']['phys_addr']
GPIO0_ADDR_range = ol.ip_dict['axi_gpio_0']['addr_range']
GPIO1_ADDR = ol.ip_dict['axi_gpio_1']['phys_addr']
GPIO1_ADDR_range = ol.ip_dict['axi_gpio_1']['addr_range']
GPIO2_ADDR = ol.ip_dict['axi_gpio_2']['phys_addr']
GPIO2_ADDR_range = ol.ip_dict['axi_gpio_2']['addr_range']
GPIO3_ADDR = ol.ip_dict['axi_gpio_3']['phys_addr']
GPIO3_ADDR_range = ol.ip_dict['axi_gpio_3']['addr_range']
GPIO4_ADDR = ol.ip_dict['axi_gpio_4']['phys_addr']
GPIO4_ADDR_range = ol.ip_dict['axi_gpio_4']['addr_range']
GPIO5_ADDR = ol.ip_dict['axi_gpio_5']['phys_addr']
GPIO5_ADDR_range = ol.ip_dict['axi_gpio_5']['addr_range']
GPIO6_ADDR = ol.ip_dict['axi_gpio_6']['phys_addr']
GPIO6_ADDR_range = ol.ip_dict['axi_gpio_6']['addr_range']
GPIO7_ADDR = ol.ip_dict['axi_gpio_7']['phys_addr']
GPIO7_ADDR_range = ol.ip_dict['axi_gpio_7']['addr_range']
GPIO8_ADDR = ol.ip_dict['axi_gpio_8']['phys_addr']
GPIO8_ADDR_range = ol.ip_dict['axi_gpio_8']['addr_range']
GPIO9_ADDR = ol.ip_dict['axi_gpio_9']['phys_addr']
GPIO9_ADDR_range = ol.ip_dict['axi_gpio_9']['addr_range']

CDMA_ADDR = ol.ip_dict['axi_cdma_0']['phys_addr']
CDMA_ADDR_range = ol.ip_dict['axi_cdma_0']['addr_range']
BRAM0_ADDR = 0xC0000000

In [43]:
KEY_0 = MMIO(GPIO1_ADDR, GPIO1_ADDR_range)
KEY_1 = MMIO(GPIO2_ADDR, GPIO2_ADDR_range)
KEY_2 = MMIO(GPIO3_ADDR, GPIO3_ADDR_range)
KEY_3 = MMIO(GPIO4_ADDR, GPIO4_ADDR_range)
START = MMIO(GPIO5_ADDR, GPIO5_ADDR_range)
ENC_DEC = MMIO(GPIO6_ADDR, GPIO6_ADDR_range)
KEY_GEN_DONE = MMIO(GPIO7_ADDR, GPIO7_ADDR_range)
DONE = MMIO(GPIO8_ADDR, GPIO8_ADDR_range)
KEY_START = MMIO(GPIO9_ADDR, GPIO9_ADDR_range)
cdma = MMIO(CDMA_ADDR, CDMA_ADDR_range)

In [44]:
NUM_IMAGES = 12
WORDS_PER_IMAGE = 76800
TOTAL_WORDS = NUM_IMAGES * WORDS_PER_IMAGE

with open("P_out.txt", "r") as f:
    lines = f.readlines()

word_list = []
for line in lines:
    line = line.strip()
    # 每行是 128-bit (32 hex)，但最右邊才是 index 小的
    for i in range(0, 32, 8):
        word = int(line[24 - i:32 - i], 16)  # 反過來切
        word_list.append(word)

packed_data = np.array(word_list, dtype=np.uint32)
input_buffer = allocate(shape=(TOTAL_WORDS,), dtype=np.uint32)
output_buffer = allocate(shape=(TOTAL_WORDS,), dtype=np.uint32)


np.copyto(input_buffer, packed_data)


In [45]:
print("格式化輸出（每行128-bit 對應4個 uint32）：")
for i in range(10):  # 只印前2行（共8個 uint32）
    words = input_buffer[i*4:(i+1)*4]
    hex_words = [f"{w:08x}" for w in words]
    print(" ".join(hex_words))


格式化輸出（每行128-bit 對應4個 uint32）：
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8
e0370734 313198a2 885a308d 3243f6a8


In [46]:
# KEY
KEY_0.write(0x0,0x09cf4f3c)
KEY_1.write(0x0,0xabf71588)
KEY_2.write(0x0,0x28aed2a6)
KEY_3.write(0x0,0x2b7e1516)

KEY_START.write(0x0, 0x1)
while (KEY_GEN_DONE.read(0x0) & 0x1) == 0:
        pass
KEY_START.write(0x0, 0x0)

In [47]:
for i in range(12):
    cdma.write(0x00, 0x4)  # Reset
    cdma.write(0x18, int(input_buffer.physical_address + i * PIXELS_PER_CHUNK * 4))  # Source from HP port
    cdma.write(0x20, BRAM0_ADDR)  # Destination to BRAM
    cdma.write(0x28, int(PIXELS_PER_CHUNK * 4))  # Total byte count
    cdma.write(0x00, 0x1)  # Start transfer

    while (cdma.read(0x04) & 0x2) == 0:
        pass
    
    ENC_DEC.write(0x0, 0x0)
    START.write(0x0, 0x1)
    
    while (DONE.read(0x0) & 0x1) == 0:
        pass
    
    START.write(0x0, 0x0)

    cdma.write(0x00, 0x4)  # Reset
    cdma.write(0x18, BRAM0_ADDR)  # Source from HP port
    cdma.write(0x20, int(output_buffer.physical_address + i * PIXELS_PER_CHUNK * 4))  # Destination to BRAM
    cdma.write(0x28, int(PIXELS_PER_CHUNK * 4))  # Total byte count
    cdma.write(0x00, 0x1)  # Start transfer
    
    while (cdma.read(0x04) & 0x2) == 0:
        pass  

In [48]:
print("格式化輸出（每行128-bit 對應4個 uint32）：")
for i in range(10):  # 只印前2行（共8個 uint32）
    words = output_buffer[i*4:(i+1)*4]
    hex_words = [f"{w:08x}" for w in words]
    print(" ".join(hex_words))

格式化輸出（每行128-bit 對應4個 uint32）：
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d
196a0b32 dc118597 02dc09fb 3925841d


In [49]:
i = 230399
words = output_buffer[i*4:(i+1)*4]
hex_words = [f"{w:08x}" for w in words]
print(" ".join(hex_words))

196a0b32 dc118597 02dc09fb 3925841d


In [ ]:
i = 230
words = output_buffer[i*4:(i+1)*4]
hex_words = [f"{w:08x}" for w in words]
print(" ".join(hex_words))
